In [3]:
import pika
import json
import uuid

def send_test_invoice():
    # 1. Define the exact credentials from your docker-compose.yml
    credentials = pika.PlainCredentials('admin', 'password')
    
    # 2. Pass them into the connection parameters
    parameters = pika.ConnectionParameters('localhost', credentials=credentials)
    connection = pika.BlockingConnection(parameters)
    channel = connection.channel()

    # Ensure the queue exists before we send
    channel.queue_declare(queue='invoice_requests', durable=True)

    # The payload the worker expects
    message = {
        "file_path": "/app/data/clean.pdf",
        "doc_id": f"TEST-RABBIT-{str(uuid.uuid4())[:8]}"
    }

    # Drop the message into the queue
    channel.basic_publish(
        exchange='',
        routing_key='invoice_requests',
        body=json.dumps(message),
        properties=pika.BasicProperties(
            delivery_mode=pika.spec.PERSISTENT_DELIVERY_MODE
        )
    )
    
    print(f"🚀 Successfully dropped payload into RabbitMQ: {message}")
    connection.close()

if __name__ == "__main__":
    send_test_invoice()

🚀 Successfully dropped payload into RabbitMQ: {'file_path': '/app/data/clean.pdf', 'doc_id': 'TEST-RABBIT-b4faf29d'}
